# Linear Regression: Your First Predictive Model

## Introduction

In Lesson 1, we prepared the Buenos Aires dataset: merged multiple CSV files, filtered to a homogeneous market segment, removed outliers, extracted geographic features, and created a clean train-test split. That preparation work exists for one purpose — to enable what happens in this lesson.

**We are going to teach a machine to predict apartment prices.**

The concept is straightforward: show the model enough labeled examples — apartments where we already know both the size and the sale price — and it will learn the mathematical relationship between them. Once it has learned that relationship, give it a new apartment's size and it will estimate the price.

This lesson focuses on **one feature** (`surface_covered_in_m2`) rather than all available features. This is intentional. Starting with a single variable lets us visualize exactly what the model is doing — we can draw the predictions as a line on a scatter plot, see where the model is right and where it is wrong, and build genuine intuition about how linear regression works before adding complexity.

> 💡 **Why start simple?** Adding more features improves a model, but it also makes it harder to reason about. When something goes wrong with a 50-feature model, it is much harder to diagnose than when something goes wrong with a 1-feature model. Starting simple, verifying it works, then adding complexity is the disciplined approach to machine learning.

### What This Lesson Covers

By the end of this lesson, you will be able to:

1. Prepare a feature matrix and target vector for the scikit-learn API
2. Split data into training and test sets and understand why each split ratio matters
3. Establish a **mean baseline** — the simplest possible prediction — as a performance benchmark
4. Train a **Linear Regression** model using scikit-learn's `.fit()` / `.predict()` pattern
5. Evaluate model performance using **MAE**, **RMSE**, and **R²** on both training and test sets
6. Diagnose **overfitting** and **underfitting** by comparing training vs. test metrics
7. Visualize model quality through predicted-vs-actual scatter plots and residual plots
8. Extract and interpret the model's learned **intercept** and **coefficient**

---

## Setup: Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169844575", h="3298dbabb7", width=700, height=450) 

## 1. Preparing the Data

Before training a model, we need to structure our data in the format scikit-learn expects: a **feature matrix** `X` and a **target vector** `y`.

For this lesson, we will work with a deliberately simple setup — one feature, one target:

- **Feature (`X`):** `surface_covered_in_m2` — the apartment's indoor area in square meters
- **Target (`y`):** `price_aprox_usd` — the sale price in US dollars

**Why only one feature?**

In Lesson 1, we engineered multiple features: `surface_covered_in_m2`, `lat`, `lon`, and multiple neighborhood indicator columns from one-hot encoding. Using all of them for the first model would make the math work, but it would obscure what the model is doing. With one feature, we can plot the entire model as a line on a scatter plot — making every concept in this lesson directly visible and verifiable.

In Lesson 3, we will bring back all the features and compare performance. That comparison will show concretely how much the additional features help.

**Code Task 2.2.1.1**

In [ ]:
# Load the data
from wrangle import clean_files
df = clean_files(...)  # <--- "./data/buenos-aires-real-estate-*.csv"
df.info()

### 1.1 Creating the Feature Matrix and Target Vector

Scikit-learn's API is consistent across all model types: every model expects `X` (features) and `y` (target) as separate inputs.

> **Feature Matrix (`X`) vs. Target Vector (`y`)**
>
> The **Feature Matrix (`X`)** must be **two-dimensional** — it should be a DataFrame (or 2D NumPy array) with rows representing observations and columns representing features. Even with only one feature, `X` must have shape `(n_rows, 1)`, not `(n_rows,)`.
>
> The **Target Vector (`y`)** is **one-dimensional** — a Series (or 1D NumPy array) with one value per observation.
>
> This two-dimensional vs. one-dimensional distinction is enforced by scikit-learn. Passing a 1D array as `X` will raise an error; passing a 2D array as `y` will raise a different error. The shapes must match the API's expectations.

**Why double brackets for X?**

`df[["surface_covered_in_m2"]]` (double brackets) returns a **DataFrame** with one column — shape `(n, 1)`. `df["surface_covered_in_m2"]` (single brackets) returns a **Series** — shape `(n,)`. Only the DataFrame form satisfies scikit-learn's requirement for a 2D feature matrix.

**Code Task 2.2.1.2**

In [ ]:
# Create the feature matrix
X = df[[...]]  # <--- df[["surface_covered_in_m2"]]

# Create the target vector
y = df[...]  # <--- df["price_aprox_usd"]

### 1.2 Train-Test Split

> **Why Split Data?**
>
> A model trained and evaluated on the *same* data would appear to perform perfectly — it has already seen every example. But that score is meaningless for real-world use, because in practice we will always predict prices for *new* listings the model was never trained on.
>
> The train-test split enforces discipline: the model learns from training data, and we measure its real performance on test data it has never seen. Only test-set metrics are honest estimates of real-world performance.
>
> - **Training set (80%):** Used for learning — the model sees these examples and adjusts its coefficients
> - **Test set (20%):** Held out entirely until evaluation — the model never uses these during training

We use `test_size=0.2` (20% test) and `random_state=42` for **reproducibility** — every run produces the same split, enabling fair comparisons across model experiments. This is the same split we prepared in Lesson 1.

**Code Task 2.2.1.3**

In [ ]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, ..., random_state=...  # <--- X, y, test_size=0.2, random_state=42
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")

**Now it is time to answer MCQ 2.2.1.1.**

## 2. Establishing a Baseline

Before building any model, we need a **baseline** — the simplest possible prediction strategy — as a performance benchmark.

### What is a baseline and why do we need one?

A baseline answers the question: *"How well could we do without any modeling at all?"* Without a baseline, we have no way to judge whether our model is actually learning anything useful.

**The mean baseline:** If someone asked you "how much does a Buenos Aires apartment cost?" and you had to give one number for every apartment regardless of size, your best answer would be the **mean price** across all training examples. This is the most naïve possible prediction — it ignores all features and simply returns the average for every single query.

> 🧠 **Why the mean, specifically?** The mean minimizes the sum of squared deviations. In other words, if you must predict the same value for everyone, the mean produces the smallest total squared error. It is the optimal constant predictor under the squared-error loss function — which is the same loss function linear regression uses. This makes it the natural baseline for comparing against a learned model.

**The baseline as a minimum bar:** Any model that performs *worse* than the mean baseline is not just unhelpful — it is actively destructive. If our linear regression model cannot beat a simple average, we have failed to extract useful signal from the data. The baseline defines the floor.

**Code Task 2.2.2.1**

In [ ]:
y_mean = ...  # <--- y_train.mean()
print("Mean house price:", y_mean)

### 2.1 Measuring Baseline Performance: MAE

Now we apply the mean as our prediction for every test apartment and measure how far off we are. The metric we use is **Mean Absolute Error (MAE)**.

**MAE: Intuition**

The Mean Absolute Error measures the average *magnitude* of prediction errors. For each test apartment, compute the absolute difference between the actual price and the predicted price (the mean). Average those absolute differences across all test apartments.

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

**Why absolute value instead of just the average difference?**

Raw differences (actual − predicted) cancel out. A $50,000 overestimate and a $50,000 underestimate would average to zero — falsely suggesting perfect accuracy. Taking the absolute value prevents this cancellation, so the MAE reflects the true average *size* of errors regardless of direction.

**Interpreting MAE in practical terms:**

A baseline MAE of $X means: *"If we predict the same average price for every apartment, we would be off by $X on average."* This is the standard of accuracy a brain-dead predictor achieves. Our trained model must do better than this to justify its complexity.

**Why use the training mean, not the test mean?**

In production, we would not know the test prices when making predictions — that is the whole point. The training mean is the only honest estimate of the population mean that is available at prediction time. Using the test mean would again be data leakage — peeking at information we would not have in the real world.

**Code Task 2.2.2.2**

In [ ]:
# Create predictions (all the same mean value) for the TEST set
y_pred_baseline = [y_mean] * len(y_test)

# Calculate Mean Absolute Error
mae_baseline = mean_absolute_error(..., ...)  # <--- mean_absolute_error(y_test, y_pred_baseline)

print("Baseline MAE:", mae_baseline)

**Now it is time to answer MCQ 2.2.2.1.**

## 3. Building the Model: Linear Regression

Now we train our first actual predictive model. Linear regression finds the straight line through the data that minimizes prediction error — the **line of best fit**.

### What linear regression is doing

Linear regression assumes the target variable (price) can be expressed as a **linear combination** of the input features:

$$\hat{y} = \beta_0 + \beta_1 x_1$$

With one feature (apartment size):

$$\widehat{\text{price}} = \beta_0 + \beta_1 \times \text{surface\_covered\_in\_m2}$$

Where:
- $\beta_0$ is the **intercept** — the predicted price when `surface_covered_in_m2 = 0` (a mathematical anchor; the literal interpretation may not make physical sense)
- $\beta_1$ is the **coefficient** — how much the predicted price increases per additional square meter

**How the "best" line is found:**

Scikit-learn uses **Ordinary Least Squares (OLS)** to find the $\beta_0$ and $\beta_1$ that minimize the sum of squared errors between actual and predicted prices:

$$\text{minimize} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2 = \sum_{i=1}^{n} (\beta_0 + \beta_1 x_i - y_i)^2$$

OLS has a closed-form solution — there is a formula that computes the optimal $\beta$ values directly from the data. This means linear regression training is instantaneous, unlike iterative methods (like gradient descent) used in neural networks.

### The scikit-learn fit/predict pattern

Scikit-learn provides a **consistent API** for every model:

```python
# Step 1: Instantiate — create the model object with hyperparameters
model = LinearRegression()

# Step 2: Fit — train on labeled data (model learns β₀ and β₁)
model.fit(X_train, y_train)

# Step 3: Predict — generate predictions for any feature matrix
y_pred = model.predict(X_new)
```

This three-step pattern — **instantiate → fit → predict** — applies to every scikit-learn estimator: Ridge, Lasso, RandomForest, GradientBoosting, SVM. Learn it once, use it everywhere.

> 📌 **Why separate instantiation from fitting?** Separating object creation from training lets you configure the model's hyperparameters (like the regularization strength for Ridge or Lasso) before exposing it to data. It also makes the workflow explicit: model definition is separate from model training, which is separate from model evaluation.

> ⚠️ **A common mistake:** Calling `.fit()` on test data. The model should only ever see training data during `.fit()`. The test set is reserved entirely for evaluation.

**Code Task 2.2.3.1**

In [ ]:
from IPython.display import VimeoVideo

# Bigger video
VimeoVideo("1169844382", h="3298dbabb7", width=700, height=450) 

**Code Task 2.2.3.1**: Instantiate a `LinearRegression` model named `model_lr` and fit it to your training data (`X_train` and `y_train`).

In [ ]:
# Instantiate the model
model_lr = ...  # <--- LinearRegression()

# Fit the model
model_lr.fit(..., ...)  # <--- model_lr.fit(X_train, y_train)

**Now it is time to answer MCQ 2.2.3.1.**

## 4. Evaluating the Model

Training a model is only half the work. Evaluation answers the critical question: *"Did the model actually learn something useful?"*

We will measure performance on both the **training set** (data the model learned from) and the **test set** (data it has never seen). Both measurements are necessary:

- **Training metrics alone** are unreliable — a model that memorizes training data will score perfectly on training but fail on new data
- **Test metrics alone** lose information — the gap between training and test performance reveals whether the model is overfitting

### The Three Metrics We Will Use

**MAE (Mean Absolute Error):**

$$\text{MAE} = \frac{1}{n} \sum_{i=1}^{n} |y_i - \hat{y}_i|$$

The average magnitude of prediction errors in the original units (dollars). A MAE of $50,000 means the model's predictions are off by $50,000 on average. Easy to explain to non-technical stakeholders: "We are typically off by $X."

**RMSE (Root Mean Squared Error):**

$$\text{RMSE} = \sqrt{\frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2}$$

Errors are squared before averaging, then we take the square root to return to the original units. Because errors are squared, **large errors are penalized disproportionately** — a $100,000 error contributes 100 times more to RMSE than a $10,000 error. RMSE is preferred when large errors are especially undesirable.

**R² (Coefficient of Determination):**

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

The proportion of variance in the target explained by the model. R² = 1.0 means perfect predictions. R² = 0.0 means the model does no better than always predicting the mean. R² < 0 means the model is *worse* than the mean baseline.

> ⚠️ **R² caveat:** R² can be high even when predictions are systematically biased — a model that consistently overestimates by the same amount can still explain most of the variance. Always inspect residual plots alongside R² to check for systematic bias.

**Code Task 2.2.4.1**

In [ ]:
# Generate predictions on training data
y_pred_train = model_lr.predict(X_train)  # <--- model_lr.predict(X_train)

# Calculate training metrics
mae_train = mean_absolute_error(y_train, y_pred_train)  # <--- mean_absolute_error(y_train, y_pred_train)
rmse_train = root_mean_squared_error(y_train, y_pred_train)  # <--- root_mean_squared_error(y_train, y_pred_train)
r2_train = r2_score(y_train, y_pred_train)  # <--- r2_score(y_train, y_pred_train)

print(f"Training MAE: ${mae_train:,.2f}")
print(f"Training RMSE: ${rmse_train:,.2f}")
print(f"Training R²: {r2_train:.4f}")

### 4.2 Test Set Performance

> **Generalization Performance**
>
> The test set performance is the **honest measure of model quality**. It answers: "How accurately would this model predict prices for new listings the business has never seen?"
>
> Training performance measures how well the model *memorized* the training examples. Test performance measures how well it *generalized* to new ones. For a model to be useful in production, test performance must be acceptable — not just training performance.

**Code Task 2.2.4.2**

In [ ]:
# Generate predictions on test data
y_pred_test = ...  # <--- model_lr.predict(X_test)

# Calculate test metrics
mae_test = mean_absolute_error(..., ...)  # <--- mean_absolute_error(y_test, y_pred_test)
rmse_test = root_mean_squared_error(..., ...)  # <--- root_mean_squared_error(y_test, y_pred_test)
r2_test = r2_score(..., ...)  # <--- r2_score(y_test, y_pred_test)

print(f"Test MAE: ${mae_test:,.2f}")
print(f"Test RMSE: ${rmse_test:,.2f}")
print(f"Test R²: {r2_test:.4f}")

### 4.3 Diagnosing the Model: Training vs. Test Gap

The most important comparison is between training and test metrics. Three patterns are possible:

> **Overfitting (train ≪ test error)**
>
> RMSE_train is substantially lower than RMSE_test. The model has memorized the training data — it performs well on examples it has seen but generalizes poorly to new ones.
>
> **Cause:** Model is too complex relative to the data; too many features; outliers not removed; data leakage.
> **Fix:** Regularization (Lesson 3), feature selection, more training data, or stricter data cleaning.

> **Underfitting (both train and test errors are high)**
>
> Both RMSE values are large — the model fails on training data *and* test data. The model is too simple to capture the underlying relationship.
>
> **Cause:** Not enough features, wrong model family, or a real relationship that cannot be captured linearly.
> **Fix:** Add more features, try a more complex model, or engineer non-linear features.

> **Good Fit (train ≈ test, both reasonably low)**
>
> Training and test errors are close to each other and acceptably low. The model has learned a generalizable pattern, not memorized specific examples.

**About the three metrics and their tradeoffs:**

| Metric | What it measures | Sensitivity to outliers | Units |
|--------|-----------------|------------------------|-------|
| MAE | Average absolute error | Low — all errors treated equally | Dollars |
| RMSE | Root mean squared error | High — large errors penalized heavily | Dollars |
| R² | Proportion of variance explained | Moderate | Dimensionless (0–1) |

For communicating with stakeholders: **MAE** is most intuitive — "we are off by $X on average." For optimization and model selection: **RMSE** is often preferred because it penalizes large errors more harshly, matching the real cost of badly wrong predictions. **R²** gives relative context — how much better are we than always predicting the mean?

**Code Task 2.2.4.3**

In [ ]:
# Create comparison DataFrame
metrics_comparison = pd.DataFrame({
    'Metric': ['MAE', 'RMSE', 'R²'],
    'Training': [..., ..., ...],  # <--- mae_train, rmse_train, r2_train
    'Test': [..., ..., ...]  # <--- mae_test, rmse_test, r2_test
})

print(metrics_comparison)

# Calculate the difference
print(f"\nRMSE difference (Test - Train): ${rmse_test - rmse_train:,.2f}")

**Now it is time to answer MCQ 2.2.4.1.**

## 5. Communicating Results

Metrics tell us *how much* error the model makes. Visualizations tell us *where* and *why* it makes those errors. A model with acceptable RMSE might still have systematic biases that metrics alone cannot reveal.

### 5.1 Model Parameters: Intercept and Coefficient

After fitting, the model has learned two parameters that define the line of best fit: the **intercept** ($\beta_0$) and the **coefficient** ($\beta_1$).

**What they mean in our context:**

- **Intercept ($\beta_0$):** The predicted price when `surface_covered_in_m2 = 0`. The literal interpretation — an apartment of zero square meters — is nonsensical, but the intercept is mathematically necessary as the line's anchor point. Do not over-interpret its value.
- **Coefficient ($\beta_1$):** The predicted price increase per additional square meter. If $\beta_1 = 1{,}200$, the model predicts that each extra square meter of covered area adds approximately $1,200 to the price. This number is directly interpretable and useful.

**Code Task 2.2.5.1**

In [ ]:
intercept = ...  # <--- model_lr.intercept_
coefficient = ...  # <--- model_lr.coef_[0]

print(f"Intercept: ${intercept:,.2f}")
print(f"Coefficient: ${coefficient:,.2f} per square meter")

### 5.2 Predicted vs. Actual Plot

The predicted-vs-actual scatter plot is one of the most informative model diagnostics available. It compares the model's predictions (x-axis) to the true prices (y-axis).

**How to read it:**

- **Perfect predictions:** Every point falls on the 45° diagonal line (where predicted = actual)
- **Points above the diagonal:** The model *underestimated* — actual price is higher than predicted
- **Points below the diagonal:** The model *overestimated* — actual price is lower than predicted
- **Tight cluster around the diagonal:** Low prediction error overall
- **Wide scatter off the diagonal:** Large errors; the model is missing important signals

**What to look for in our plot:**

- Is the scatter roughly symmetric around the diagonal, or does the model consistently over- or underestimate in certain price ranges?
- Does the error increase at higher price points? (If so, the model struggles more with expensive apartments — a common pattern in price prediction called heteroscedasticity)
- Is there a meaningful difference between the training plot and the test plot? (A much tighter training plot suggests overfitting)

We create separate plots for training and test data to compare directly. The training plot shows how well the model learned; the test plot shows how well it generalizes.

**Code 2.2.5.2**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training set
axes[0].scatter(y_train, y_pred_train, alpha=0.5)
axes[0].plot([y_train.min(), y_train.max()],
             [y_train.min(), y_train.max()],
             'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Price (USD)')
axes[0].set_ylabel('Predicted Price (USD)')
axes[0].set_title('Training Set: Predicted vs Actual')
axes[0].legend()

# Test set
axes[1].scatter(y_test, y_pred_test, alpha=0.5, color='green')
axes[1].plot([y_test.min(), y_test.max()],
             [y_test.min(), y_test.max()],
             'r--', lw=2, label='Perfect Prediction')
axes[1].set_xlabel('Actual Price (USD)')
axes[1].set_ylabel('Predicted Price (USD)')
axes[1].set_title('Test Set: Predicted vs Actual')
axes[1].legend()

plt.tight_layout()

### 5.3 Residual Analysis

A **residual** is the difference between the actual value and the model's prediction:

$$e_i = y_i - \hat{y}_i$$

If the model predicted $280,000 but the actual price was $320,000, the residual is $+40,000 (positive = underestimated). If it predicted $350,000 for a $300,000 apartment, the residual is $-50,000 (negative = overestimated).

**Why residuals matter:**

Residual plots reveal patterns that aggregate metrics like MAE and RMSE hide. A model might have an acceptable average error while still systematically underestimating expensive apartments or overestimating cheap ones. Those systematic errors indicate that the model is missing an important relationship in the data.

**The ideal residual plot:** Residuals randomly scattered around zero with no discernible pattern — just noise. This means the model has captured all the systematic structure and the remaining errors are unpredictable.

**Code Task 2.2.5.2**

In [ ]:
residuals_train = y_train - ...  # <--- y_train - y_pred_train
residuals_test = ... - y_pred_test  # <--- y_test - y_pred_test

fig, axes = plt.subplots(1, 2, figsize=(9, 6))

# Training residuals
axes[0].scatter(y_pred_train, residuals_train, alpha=0.5)
axes[0].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0].set_xlabel('Predicted Price (USD)')
axes[0].set_ylabel('Residuals (USD)')
axes[0].set_title('Training Set: Residual Plot')

# Test residuals
axes[1].scatter(y_pred_test, residuals_test, alpha=0.5, color='green')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Price (USD)')
axes[1].set_ylabel('Residuals (USD)')
axes[1].set_title('Test Set: Residual Plot')

plt.tight_layout()

> **Interpreting Residual Plots**
>
> **Good pattern:** Residuals randomly scattered around zero — no curve, no funnel, no systematic trend. This is what we are hoping to see.
>
> **Bad pattern — curved residuals:** The residuals form a curve (high for small and large predicted values, low in the middle, or vice versa). This indicates a **non-linear relationship** that a linear model cannot capture. Consider adding polynomial features or using a non-linear model.
>
> **Bad pattern — funnel shape (heteroscedasticity):** The spread of residuals increases as predicted values increase. For expensive apartments, the model's errors are much larger in absolute terms than for cheap apartments. This is common in price prediction data: a 10% pricing error on a $100,000 apartment ($10,000) is in the same proportional range as a 10% error on a $400,000 apartment ($40,000), but the absolute difference is four times larger.
>
> Heteroscedasticity violates a key assumption of OLS: that errors have constant variance. It means that confidence intervals around predictions are unreliable — they may be too narrow for expensive apartments and too wide for cheap ones. We will revisit this in Lesson 4.
>
> **Bad pattern — systematic bias:** Residuals are consistently positive (model always underestimates) or consistently negative (model always overestimates). This suggests the model's functional form is wrong — it is not the right type of equation for this relationship.

### 5.4 Line of Best Fit

The final visualization places the model's predictions as a line over the original scatter plot — this is the "line of best fit" that linear regression has learned from the training data.

**What the line shows:** For every possible value of `surface_covered_in_m2`, the line shows the model's predicted price. The slope of the line is the coefficient $\beta_1$; the y-intercept is $\beta_0$.

**What to notice:** How well does the line thread through the cloud of points? Are there systematic deviations (e.g., does the line overestimate for very small apartments and underestimate for large ones)? A perfect linear fit would have all points balanced symmetrically around the line.

**Code 2.2.5.4**

In [ ]:
fig, ax = plt.subplots()
ax.scatter(X_train, y_train, alpha=0.5, label='Training Data')
ax.scatter(X_test, y_test, alpha=0.5, color='green', label='Test Data')

# Create evenly-spaced x values for the line
x_line = np.linspace(X_train.min(), X_train.max(), 100).reshape(-1, 1)
ax.plot(x_line, model_lr.predict(x_line), color="red", linewidth=2, label="Linear Model")

ax.set_xlabel("Surface Area [sq meters]")
ax.set_ylabel("Price [USD]")
ax.set_title("Buenos Aires: Price vs. Surface Area")
ax.legend()

## Summary

In this lesson, you built, evaluated, and diagnosed your first machine learning model. Here is what each step accomplished and what it revealed:

| Step | What we did | Key insight |
|------|-------------|-------------|
| **Data prep** | Single-feature X/y split, 80/20 train-test | One feature for interpretability; test set for honest evaluation |
| **Baseline** | Mean price as constant prediction | Defines the floor — any model that cannot beat this is useless |
| **Train model** | `LinearRegression().fit(X_train, y_train)` | OLS finds the β₀ and β₁ that minimize squared errors |
| **Evaluate** | MAE, RMSE, R² on train and test | Three metrics give complementary views of error magnitude |
| **Diagnose** | Train vs. test gap | Gap reveals overfitting; similar values indicate good generalization |
| **Visualize** | Predicted-vs-actual, residual plots | Reveals *where* and *why* the model makes errors |
| **Interpret** | Intercept and coefficient | β₁ = price per m² — a directly actionable business number |

**Key Takeaways:**

- **The baseline is the minimum bar** — a model that cannot beat the mean average is not worth deploying
- **The fit/predict pattern** — `instantiate → fit → predict` — is scikit-learn's universal API; master it once, use it for every model
- **MAE vs. RMSE:** MAE is easier to explain; RMSE penalizes large errors more heavily and is preferred when big mistakes are costly
- **R² has a useful interpretation** but must be paired with residual plots to check for systematic bias
- **A single feature is not enough** — the residual patterns we observed indicate that size alone cannot fully explain apartment prices; neighborhood, location, and other characteristics also matter
- **This model motivates what comes next** — Lesson 3 will bring back all features and introduce regularization to handle the complexity

---

## Discussion Questions

1. Our model uses only one feature (apartment size). Looking at the residual plots, what patterns do you observe that suggest size alone is insufficient? What other features might explain the residuals?

2. Compare the baseline MAE to the linear regression MAE. How much of an improvement did the model make? Is this improvement meaningful in practical terms?

3. The intercept value represents the "predicted price for a 0 m² apartment." Why does this not have a useful literal interpretation, yet is still mathematically necessary?

4. What does heteroscedasticity in the residuals imply about the reliability of this model's predictions for expensive vs. cheap apartments?

5. If you were presenting this model to a real estate company, how would you explain the coefficient $\beta_1$ in plain English? What would you say about its limitations?

➡️ **In Lesson 3**, we will bring back all features — size, coordinates, and neighborhood indicators — and introduce **regularization** (Ridge and Lasso) to prevent overfitting with a large feature set. We will also build these models inside a scikit-learn **Pipeline** to prevent data leakage from the one-hot encoding step.